<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/CH7_Agent_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
from google.colab import userdata

# Groq API 키 설정
YOUR_API_KEY = userdata.get('groq')
os.environ["GROQ_API_KEY"] = YOUR_API_KEY

In [3]:
!pip install -qU langchain-groq
!pip install -qU langchain-community
!pip install duckduckgo-search -q
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 7.9 MB/s eta 0:00:00


In [4]:
# @title **Reflection**

from typing import Annotated, TypedDict

from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage,
)
from langchain_groq import ChatGroq
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

# Initialize chat model
model = ChatGroq(model="llama-3.3-70b-versatile")

# Define state type
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# Define prompts
generate_prompt = SystemMessage(
    "You are an essay assistant tasked with writing excellent 3-paragraph essays."
    " Generate the best essay possible for the user's request."
    " If the user provides critique, respond with a revised version of your previous attempts."
)

reflection_prompt = SystemMessage(
    "You are a teacher grading an essay submission. Generate critique and recommendations for the user's submission."
    " Provide detailed recommendations, including requests for length, depth, style, etc."
)

def generate(state: State) -> State:
    answer = model.invoke([generate_prompt] + state["messages"])
    return {"messages": [answer]}

def reflect(state: State) -> State:
    # Invert the messages to get the LLM to reflect on its own output
    cls_map = {AIMessage: HumanMessage, HumanMessage: AIMessage}
    # First message is the original user request. We hold it the same for all nodes
    translated = [reflection_prompt, state["messages"][0]] + [
        cls_map[msg.__class__](content=msg.content) for msg in state["messages"][1:]
    ]
    answer = model.invoke(translated)
    # We treat the output of this as human feedback for the generator
    return {"messages": [HumanMessage(content=answer.content)]}

def should_continue(state: State):
    if len(state["messages"]) > 6:
        # End after 3 iterations, each with 2 messages
        return END
    else:
        return "reflect"

# Build the graph
builder = StateGraph(State)
builder.add_node("generate", generate)
builder.add_node("reflect", reflect)
builder.add_edge(START, "generate")
builder.add_conditional_edges("generate", should_continue)
builder.add_edge("reflect", "generate")

graph = builder.compile()

# Example usage
initial_state = {
    "messages": [
        HumanMessage(
            content="Write an essay about the relevance of 'The Little Prince' today."
        )
    ]
}

# Run the graph
for output in graph.stream(initial_state):
    message_type = "generate" if "generate" in output else "reflect"
    print("\nNew message:", output[message_type]
          ["messages"][-1].content[:100], "...")


New message: The Little Prince, written by Antoine de Saint-Exupéry, has been a timeless classic since its public ...

New message: **Overall Assessment:**
The essay provides a clear and well-structured argument for the relevance of ...

New message: I appreciate the detailed feedback and assessment of my essay. Based on the recommendations, I have  ...

New message: **Overall Assessment:**
The revised essay demonstrates significant improvement in addressing the wea ...

New message: I appreciate the detailed feedback and assessment of my revised essay. Based on the recommendations, ...

New message: **Overall Assessment:**
The revised essay demonstrates significant improvement in addressing the wea ...

New message: I appreciate the detailed feedback and assessment of my revised essay. Based on the recommendations, ...


In [14]:
[generate_prompt] + initial_state["messages"]

[SystemMessage(content="You are an essay assistant tasked with writing excellent 3-paragraph essays. Generate the best essay possible for the user's request. If the user provides critique, respond with a revised version of your previous attempts.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="Write an essay about the relevance of 'The Little Prince' today.", additional_kwargs={}, response_metadata={}, id='e4a1814d-3bee-4959-a524-46ede299725e')]

In [13]:
model.invoke([generate_prompt] + initial_state["messages"])

AIMessage(content="The Little Prince, written by Antoine de Saint-Exupéry, is a timeless novella that has captivated readers of all ages with its poignant and thought-provoking themes. Despite being written over 75 years ago, the story remains remarkably relevant today, offering insights into the human condition that are just as pertinent now as they were when the book was first published. One of the primary reasons for its enduring relevance is its exploration of the complexities of adult relationships and the way we often lose sight of what truly matters in life. The novella's protagonist, a young prince from a distant asteroid, observes the strange behaviors of the adults he encounters on his journey, highlighting the ways in which we can become disconnected from our emotions, our relationships, and the natural world.\n\nThe Little Prince's themes of love, friendship, and the importance of human connection are also highly relevant in today's world. In an era dominated by technology 

In [6]:
# 카운터 변수 초기화
gen_count = 0
ref_count = 0

print("🚀 [성찰(Reflection) 에이전트 프로세스 시작]\n")

for output in graph.stream(initial_state):
    # 1. 작성/수정 노드(generate) 출력 처리
    if "generate" in output:
        gen_count += 1
        content = output["generate"]["messages"][-1].content

        if gen_count == 1:
            print("📝 [1차 초안 작성]")
            print("=" * 60)
        elif gen_count <= 3:
            print(f"✏️ [{gen_count - 1}차 피드백 반영 - {gen_count}차 수정본]")
            print("=" * 60)
        else:
            print("🎉 [최종 완결본 (Final Version)]")
            print("=" * 60)

        print(content)
        print("\n")

    # 2. 피드백/검수 노드(reflect) 출력 처리
    elif "reflect" in output:
        ref_count += 1
        content = output["reflect"]["messages"][-1].content

        print("-" * 60)
        print(f"🔍 [교사의 {ref_count}차 피드백 및 지적사항 (Critique #{ref_count})]")
        print("-" * 60)
        print(content)
        print("\n")

print("✅ [프로세스 완료]")

🚀 [성찰(Reflection) 에이전트 프로세스 시작]

📝 [1차 초안 작성]
The Little Prince, written by Antoine de Saint-Exupéry in 1943, is a timeless novella that continues to captivate readers of all ages with its poignant and thought-provoking themes. Despite being written over seven decades ago, the story remains remarkably relevant today, offering valuable insights into the human condition, relationships, and the importance of empathy and understanding. The narrative follows a young prince's journey as he travels from his own small planet to Earth, encountering various characters who embody the flaws and contradictions of adult society. Through the prince's innocent and curious perspective, Saint-Exupéry critiques the superficiality and materialism of modern life, highlighting the need for genuine connections and meaningful relationships in a world that often prioritizes ambition and status over people and emotions.

One of the key reasons why The Little Prince remains relevant today is its ability to trans

# **Subgraphs in LangGraph**

In [17]:
# @title **Calling a Subgraph Directly**
from typing import TypedDict

from langgraph.graph import START, StateGraph


# Define the state types for parent and subgraph
class State(TypedDict):
    foo: str  # this key is shared with the subgraph


class SubgraphState(TypedDict):
    foo: str  # this key is shared with the parent graph
    bar: str


# Define subgraph
def subgraph_node(state: SubgraphState):
    # note that this subgraph node can communicate with the parent graph via the shared "foo" key
    return {"foo": state["foo"] + "bar"}


subgraph_builder = StateGraph(SubgraphState)
subgraph_builder.add_node("subgraph_node", subgraph_node)
subgraph_builder.add_edge(START, "subgraph_node") # Added this line
# Additional subgraph setup would go here
subgraph = subgraph_builder.compile()

# Define parent graph
builder = StateGraph(State)
builder.add_node("subgraph", subgraph)
builder.add_edge(START, "subgraph")
# Additional parent graph setup would go here
graph = builder.compile()

# Example usage
initial_state = {"foo": "hello"}
result = graph.invoke(initial_state)
print(f"Result: {result}")  # Should append "bar" to the foo value

Result: {'foo': 'hellobar'}
